# In-Class Competition: Home Credit Default Risk
## Advanced Feature Engineering & Cross-Validated GBDT Pipeline

This notebook contains an ablation-controlled, production-grade pipeline targeting the ROC-AUC evaluation metric.

### Pipeline Highlights:
- **Complete Feature Set:** Preserves all 83 domain, ratio, age, car, external source, frequency, and group-mean statistics.
- **High-Value Additions:** Optional missingness indicators (`MISSING_COUNT`) and document submission completeness.
- **Proper GBDT Regularization:** Active row bagging (`subsample_freq=1`), optimized `min_child_samples`, and controlled `num_leaves` to reduce leaf overfitting.
- **Fold Integrity:** Leak-free 5-fold Stratified Cross-Validation with clean validation-only early stopping.
- **Ensembling (Optional):** Rank-averaged blending with CatBoost, with blend weights mathematically optimized on Out-Of-Fold predictions.


### 1. Environment & Dataset Auto-Detection
Automatically handles Kaggle, Google Colab, and local environments.


In [ ]:
import sys
import os
import glob
from pathlib import Path

# Kaggle automatic dataset locator
if os.path.exists('/kaggle'):
    train_files = glob.glob('/kaggle/input/**/train.csv', recursive=True)
    if train_files:
        source_dir = os.path.dirname(train_files[0])
        os.system(f'rm -rf ./input && ln -s "{source_dir}" ./input')
        print(f'[Kaggle] Linked dataset from: {source_dir} -> ./input')
    else:
        print('[Kaggle] No dataset attached yet. Please upload input files via right sidebar.')
elif 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd '/content/drive/MyDrive/competition'
else:
    print('[Local] Running in local environment.')

!ls -lh input


### 2. Imports & Settings


In [ ]:
import gc
import time
import zipfile
import warnings
from scipy.stats import rankdata
from scipy.optimize import minimize_scalar

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb

warnings.filterwarnings('ignore')
print(f"NumPy: {np.__version__}, Pandas: {pd.__version__}, LightGBM: {lgb.__version__}")


### 3. Experiment Toggles (Ablation against 0.75849 baseline)
Easily turn improvements on or off to verify their exact impact on Out-Of-Fold ROC-AUC.


In [ ]:
# ===========================================================================
# EXPERIMENT TOGGLES
# ===========================================================================
ENABLE_PARAM_FIXES    = True   # True: subsample_freq=1, min_child_samples=60, num_leaves=24
ENABLE_EXTRA_FEATURES = True   # True: adds MISSING_COUNT & document count (keeps all 83 original)
SEEDS                 = [42]   # Set to [42, 2024] to test multi-seed averaging
ENABLE_CATBOOST       = False  # Set to True to train CatBoost & find optimal blend weight on OOF
# ===========================================================================


### 4. Data Loading


In [ ]:
current_dir = Path.cwd()
input_dir = current_dir / 'input'

train_path = input_dir / 'train.csv'
test_path = input_dir / 'test.csv'
sample_sub_path = input_dir / 'sample_submission.csv'

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample_sub = pd.read_csv(sample_sub_path)

y = train['TARGET'].values
test_ids = test['SK_ID_CURR']

print(f"Train shape: {train.shape}, Test shape: {test.shape}")
print(f"Target default rate: {y.mean():.4%}")


### 5. Feature Engineering
Builds all core domain ratios, EXT_SOURCE interactions, per-capita quantities, and optional missingness signals.


In [ ]:
def engineer_features(df):
    df = df.copy()
    
    # Anomaly indicator and replacement
    df['DAYS_EMPLOYED_ANOM'] = (df['DAYS_EMPLOYED'] == 365243).astype(int)
    df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)
    
    # Credit and income ratios
    df['PAYMENT_RATE'] = df['AMT_ANNUITY'] / df['AMT_CREDIT']
    df['CREDIT_INCOME_RATIO'] = df['AMT_CREDIT'] / (df['AMT_INCOME_TOTAL'] + 1)
    df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY'] / (df['AMT_INCOME_TOTAL'] + 1)
    df['INCOME_CREDIT_DIFF'] = df['AMT_INCOME_TOTAL'] - df['AMT_CREDIT']
    df['GOODS_CREDIT_RATIO'] = df['AMT_GOODS_PRICE'] / (df['AMT_CREDIT'] + 1)
    df['GOODS_CREDIT_DIFF'] = df['AMT_GOODS_PRICE'] - df['AMT_CREDIT']
    
    # Per-person family ratios
    fam_members = df['CNT_FAM_MEMBERS'].fillna(1).replace(0, 1)
    df['INCOME_PER_PERSON'] = df['AMT_INCOME_TOTAL'] / fam_members
    df['CREDIT_PER_PERSON'] = df['AMT_CREDIT'] / fam_members
    df['ANNUITY_PER_PERSON'] = df['AMT_ANNUITY'] / fam_members
    
    # Time and age ratios
    df['DAYS_EMPLOYED_PERC'] = df['DAYS_EMPLOYED'] / df['DAYS_BIRTH']
    df['AGE_YEARS'] = -df['DAYS_BIRTH'] / 365.25
    df['EMPLOYED_YEARS'] = -df['DAYS_EMPLOYED'] / 365.25
    df['REGISTRATION_BIRTH_RATIO'] = df['DAYS_REGISTRATION'] / df['DAYS_BIRTH']
    df['ID_PUBLISH_BIRTH_RATIO'] = df['DAYS_ID_PUBLISH'] / df['DAYS_BIRTH']
    df['PHONE_BIRTH_RATIO'] = df['DAYS_LAST_PHONE_CHANGE'] / df['DAYS_BIRTH']
    df['PHONE_EMPLOYED_RATIO'] = df['DAYS_LAST_PHONE_CHANGE'] / (df['DAYS_EMPLOYED'] - 1)
    
    # Car features
    df['FLAG_OWN_CAR'] = df['OWN_CAR_AGE'].notnull().astype(int)
    df['CAR_EMPLOYED_RATIO'] = (df['OWN_CAR_AGE'] * 365) / (-df['DAYS_EMPLOYED'] + 1)
    df['CAR_BIRTH_RATIO'] = (df['OWN_CAR_AGE'] * 365) / (-df['DAYS_BIRTH'] + 1)
    
    # External source combinations
    ext_sources = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
    df['EXT_SOURCES_MEAN'] = df[ext_sources].mean(axis=1)
    df['EXT_SOURCES_STD'] = df[ext_sources].std(axis=1)
    df['EXT_SOURCES_MIN'] = df[ext_sources].min(axis=1)
    df['EXT_SOURCES_MAX'] = df[ext_sources].max(axis=1)
    df['EXT_SOURCES_PROD'] = df['EXT_SOURCE_1'] * df['EXT_SOURCE_2'] * df['EXT_SOURCE_3']
    df['EXT_1_x_EXT_2'] = df['EXT_SOURCE_1'] * df['EXT_SOURCE_2']
    df['EXT_2_x_EXT_3'] = df['EXT_SOURCE_2'] * df['EXT_SOURCE_3']
    df['EXT_1_x_EXT_3'] = df['EXT_SOURCE_1'] * df['EXT_SOURCE_3']
    
    df['EXT_1_over_BIRTH'] = df['EXT_SOURCE_1'] / df['DAYS_BIRTH']
    df['EXT_2_over_BIRTH'] = df['EXT_SOURCE_2'] / df['DAYS_BIRTH']
    df['EXT_3_over_BIRTH'] = df['EXT_SOURCE_3'] / df['DAYS_BIRTH']
    df['EXT_2_over_CREDIT'] = df['EXT_SOURCE_2'] / (df['AMT_CREDIT'] + 1)
    df['EXT_3_over_CREDIT'] = df['EXT_SOURCE_3'] / (df['AMT_CREDIT'] + 1)
    
    # Contacts & Bureau
    df['FLAG_CONTACTS_SUM'] = df['FLAG_MOBIL'] + df['FLAG_EMP_PHONE'] + df['FLAG_WORK_PHONE']
    bureau_cols = ['AMT_REQ_CREDIT_BUREAU_HOUR', 'AMT_REQ_CREDIT_BUREAU_MON', 
                   'AMT_REQ_CREDIT_BUREAU_QRT', 'AMT_REQ_CREDIT_BUREAU_YEAR']
    df['BUREAU_ENQUIRIES_SUM'] = df[bureau_cols].sum(axis=1)
    
    # Extra Missingness & Document signals (enabled via toggle)
    if ENABLE_EXTRA_FEATURES:
        df['MISSING_COUNT'] = df.isnull().sum(axis=1)
        doc_cols = [c for c in df.columns if c.startswith('FLAG_DOCUMENT_')]
        if doc_cols:
            df['DOCUMENTS_PROVIDED_COUNT'] = df[doc_cols].sum(axis=1)
    
    return df

print("Building engineered features...")
train_fe = engineer_features(train)
test_fe = engineer_features(test)

# Frequency encoding & group statistics (original 83-feature pipeline)
combined = pd.concat([train_fe, test_fe], axis=0, ignore_index=True)
freq_cols = ['ORGANIZATION_TYPE', 'OCCUPATION_TYPE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE']
for col in freq_cols:
    freq = combined[col].value_counts(normalize=True)
    train_fe[f'{col}_FREQ'] = train_fe[col].map(freq)
    test_fe[f'{col}_FREQ'] = test_fe[col].map(freq)

for grp_col in ['NAME_EDUCATION_TYPE', 'OCCUPATION_TYPE']:
    for num_col in ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'EXT_SOURCE_2']:
        grp_stats = combined.groupby(grp_col)[num_col].agg(['mean']).reset_index()
        grp_stats.columns = [grp_col, f'{grp_col}_{num_col}_MEAN']
        train_fe = train_fe.merge(grp_stats, on=grp_col, how='left')
        test_fe = test_fe.merge(grp_stats, on=grp_col, how='left')
        train_fe[f'{num_col}_RATIO_{grp_col}'] = train_fe[num_col] / (train_fe[f'{grp_col}_{num_col}_MEAN'] + 1)
        test_fe[f'{num_col}_RATIO_{grp_col}'] = test_fe[num_col] / (test_fe[f'{grp_col}_{num_col}_MEAN'] + 1)

print(f"Total features engineered: {train_fe.shape[1] - 2}")


### 6. Categorical Encoding
Uses `pd.Categorical` to ensure seamless native category handling in LightGBM.


In [ ]:
ignore_cols = ['SK_ID_CURR', 'TARGET']
features = [c for c in train_fe.columns if c not in ignore_cols]
cat_cols = train_fe[features].select_dtypes(include=['object', 'category']).columns.tolist()

for c in cat_cols:
    le = LabelEncoder()
    unique_vals = list(train_fe[c].astype(str).unique()) + list(test_fe[c].astype(str).unique())
    le.fit(unique_vals)
    train_fe[c] = pd.Categorical(le.transform(train_fe[c].astype(str)))
    test_fe[c] = pd.Categorical(le.transform(test_fe[c].astype(str)))

X = train_fe[features]
X_test = test_fe[features]
print(f"Data ready: {X.shape[1]} features, {len(X)} rows")


### 7. Model Training (5-Fold Stratified Cross-Validation)
Trains LightGBM with active bagging and evaluates on validation splits.


In [ ]:
if ENABLE_PARAM_FIXES:
    print('>> Using Regularized LightGBM (subsample_freq=1, min_child=60, num_leaves=24)')
    lgb_params = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'learning_rate': 0.02,
        'num_leaves': 24,
        'max_depth': -1,
        'min_child_samples': 60,
        'subsample': 0.8,
        'subsample_freq': 1,
        'colsample_bytree': 0.65,
        'reg_alpha': 0.5,
        'reg_lambda': 10.0,
        'n_estimators': 2500,
        'verbose': -1,
        'n_jobs': -1
    }
else:
    print('>> Using Original LightGBM baseline parameters')
    lgb_params = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'learning_rate': 0.03,
        'num_leaves': 31,
        'max_depth': -1,
        'min_child_samples': 20,
        'subsample': 0.8,
        'colsample_bytree': 0.7,
        'reg_alpha': 0.1,
        'reg_lambda': 10.0,
        'n_estimators': 1500,
        'verbose': -1,
        'n_jobs': -1
    }

n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

total_oof_lgb = np.zeros(len(X))
total_test_lgb = np.zeros(len(X_test))
feature_importances = pd.DataFrame(index=features)

for s_idx, seed in enumerate(SEEDS):
    print(f"\n===== LIGHTGBM SEED {seed} ({s_idx + 1}/{len(SEEDS)}) =====")
    current_params = lgb_params.copy()
    current_params['random_state'] = seed
    
    seed_oof = np.zeros(len(X))
    seed_test = np.zeros(len(X_test))
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, y_tr = X.iloc[train_idx], y[train_idx]
        X_va, y_va = X.iloc[val_idx], y[val_idx]
        
        model = lgb.LGBMClassifier(**current_params)
        callbacks = [lgb.early_stopping(stopping_rounds=100, verbose=False)]
        
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            eval_names=['valid'],
            categorical_feature=cat_cols,
            callbacks=callbacks
        )
        
        val_pred = model.predict_proba(X_va)[:, 1]
        seed_oof[val_idx] = val_pred
        seed_test += model.predict_proba(X_test)[:, 1] / n_splits
        
        fold_auc = roc_auc_score(y_va, val_pred)
        print(f"Fold {fold + 1} validation AUC: {fold_auc:.5f} (best iteration: {model.best_iteration_})")
        feature_importances[f'fold_{fold + 1}_s{seed}'] = model.feature_importances_
        
    total_oof_lgb += seed_oof / len(SEEDS)
    total_test_lgb += seed_test / len(SEEDS)

auc_lgb = roc_auc_score(y, total_oof_lgb)
print('-' * 50)
print(f"Overall LightGBM OOF ROC-AUC: {auc_lgb:.5f}  (Baseline was 0.75849)")
print('-' * 50)

final_oof_preds = total_oof_lgb
final_test_preds = total_test_lgb


### 8. Optional CatBoost Training & Rank-Averaged Blend
Enabled when `ENABLE_CATBOOST = True`. Finds optimal blend weights on Out-Of-Fold predictions.


In [ ]:
if ENABLE_CATBOOST:
    print("\n===== TRAINING CATBOOST =====")
    from catboost import CatBoostClassifier
    
    oof_cb = np.zeros(len(X))
    test_cb = np.zeros(len(X_test))
    
    X_cb = X.copy()
    X_test_cb = X_test.copy()
    for c in cat_cols:
        X_cb[c] = X_cb[c].astype(str).fillna('missing')
        X_test_cb[c] = X_test_cb[c].astype(str).fillna('missing')
    cb_cat_indices = [features.index(c) for c in cat_cols]

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        cb_model = CatBoostClassifier(
            iterations=1500,
            learning_rate=0.03,
            depth=6,
            eval_metric='AUC',
            random_seed=42 + fold,
            early_stopping_rounds=100,
            verbose=False,
            task_type='CPU'  # change to GPU if GPU accelerator is on
        )
        cb_model.fit(
            X_cb.iloc[train_idx], y[train_idx],
            eval_set=(X_cb.iloc[val_idx], y[val_idx]),
            cat_features=cb_cat_indices
        )
        val_pred = cb_model.predict_proba(X_cb.iloc[val_idx])[:, 1]
        oof_cb[val_idx] = val_pred
        test_cb += cb_model.predict_proba(X_test_cb)[:, 1] / n_splits
        print(f"CatBoost Fold {fold + 1} AUC: {roc_auc_score(y[val_idx], val_pred):.5f}")

    auc_cb = roc_auc_score(y, oof_cb)
    print(f"CatBoost OOF AUC: {auc_cb:.5f}")

    # Rank standardization
    r_lgb = rankdata(total_oof_lgb) / len(total_oof_lgb)
    r_cb = rankdata(oof_cb) / len(oof_cb)
    
    # Optimize blend weight strictly on OOF ROC-AUC
    def loss_fn(w):
        blend = w * r_lgb + (1 - w) * r_cb
        return -roc_auc_score(y, blend)
    
    res = minimize_scalar(loss_fn, bounds=(0, 1), method='bounded')
    best_w = res.x
    best_blend_auc = -res.fun
    
    print(f">> Optimal LightGBM Weight: {best_w:.3f} | CatBoost Weight: {1 - best_w:.3f}")
    print(f">> Blended OOF AUC: {best_blend_auc:.5f} (Gain over LGBM: {best_blend_auc - auc_lgb:+.5f})")
    
    r_test_lgb = rankdata(total_test_lgb) / len(total_test_lgb)
    r_test_cb = rankdata(test_cb) / len(test_cb)
    final_test_preds = best_w * r_test_lgb + (1 - best_w) * r_test_cb
else:
    print('CatBoost ensembling is disabled (set ENABLE_CATBOOST = True to enable).')


### 9. Feature Importance Analysis


In [ ]:
feature_importances['mean_importance'] = feature_importances.mean(axis=1)
top25 = feature_importances['mean_importance'].sort_values(ascending=False).head(25)

plt.figure(figsize=(10, 8))
sns.barplot(x=top25.values, y=top25.index)
plt.title('Top 25 Features by Importance')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()


### 10. Save Submission & Create Reproducible Honors Archive


In [ ]:
output_dir = current_dir / 'output'
os.makedirs(output_dir, exist_ok=True)
sub_file = output_dir / 'submission.csv'

submission = pd.DataFrame({
    'SK_ID_CURR': test_ids,
    'TARGET': final_test_preds
})
submission.to_csv(sub_file, index=False)
print(f"Saved final submission file to: {sub_file}")
print(f"Submission shape: {submission.shape}")
print(submission.head())

# Create zip archive for Honors requirement
nb_name = 'champion_solution.ipynb'
zip_file = current_dir / 'champion_solution.zip'

nb_file = current_dir / nb_name
with zipfile.ZipFile(zip_file, 'w', zipfile.ZIP_DEFLATED) as zf:
    if nb_file.exists():
        zf.write(nb_file, arcname=nb_name)

print(f"Created Honors zip file: {zip_file}")


### 11. Download Outputs (Works seamlessly on Kaggle & Colab)


In [ ]:
from IPython.display import FileLink, display

if 'google.colab' in sys.modules:
    from google.colab import files
    files.download(str(sub_file))
    files.download(str(zip_file))
else:
    print('Click the links below to download your files:')
    display(FileLink('output/submission.csv'))
    display(FileLink('champion_solution.zip'))
